In [1]:
import json
import pandas as pd
import re

# Golden set load karo
golden_set = []
with open("../data/schemas/golden_set_final.jsonl", encoding="utf-8") as f:
    for line in f:
        golden_set.append(json.loads(line))

golden_conv_ids = set(ex["conversation_id"] for ex in golden_set)
print(f"Golden set size: {len(golden_set)}, unique conv_ids: {len(golden_conv_ids)}")

# Saari clean conversations load karo
conversations = []
with open("../data/processed/amazon_conversations_clean.jsonl", encoding="utf-8") as f:
    for line in f:
        conversations.append(json.loads(line))

print(f"Total clean conversations: {len(conversations)}")

Golden set size: 211, unique conv_ids: 211
Total clean conversations: 60669


In [2]:
dev_pool = [c for c in conversations if c["conversation_id"] not in golden_conv_ids]
print(f"Dev pool (golden excluded): {len(dev_pool)}")

Dev pool (golden excluded): 60458


In [3]:
dev_first_msgs = []
for conv in dev_pool:
    first_turn = next((t for t in conv["turns"] if t["role"] == "CUSTOMER"), None)
    if first_turn and len(first_turn["cleaned_text"].strip()) > 0:
        dev_first_msgs.append({
            "conversation_id": conv["conversation_id"],
            "text": first_turn["cleaned_text"],
        })

df_dev = pd.DataFrame(dev_first_msgs)
print(f"Dev messages: {len(df_dev)}")

Dev messages: 60458


In [4]:
taxonomy_patterns = {
    "delivery_status": [r"\bdeliver", r"\bpackage\b", r"\btrack", r"\barriv", r"\bshipp", r"\bwhere is\b", r"\breceived?\b", r"\brcvd\b"],
    "order_issue": [r"\border\b", r"\bordered\b", r"\bcancel", r"\bwrong item\b"],
    "refund_return": [r"\brefund", r"\breturn", r"\bmoney back\b", r"\bexchange\b", r"\bcashback\b"],
    "payment_billing": [r"\bcharge", r"\bpayment\b", r"\bbill", r"\bcard\b"],
    "pricing_query": [r"\bmrp\b", r"\bprice\b", r"\bpricing\b", r"\boverchar", r"\bcost\b"],
    "prime_membership": [r"\bprime\b", r"\bmembership\b", r"\bsubscription\b"],
    "content_streaming_issue": [r"\bprime video\b", r"\baudible\b", r"\bsubtitle", r"\bepisode\b", r"\bgeo setting", r"\bregion\b", r"\bstream"],
    "device_tech_support": [r"\becho\b", r"\balexa\b", r"\bkindle\b", r"\bwifi\b", r"\bapp\b", r"\bdevice\b", r"\bconnect", r"\bupdate\b"],
    "repair_service_status": [r"\brepair\b", r"\bservice cent", r"\bsubmitted\b.*\b(mobile|device|phone)\b"],
    "account_access": [r"\baccount\b", r"\bpassword\b", r"\blogin\b", r"\block", r"\baccess\b"],
    "availability_question": [r"\bwhen (will|would)\b", r"\bavailable\b", r"\bavailability\b"],
    "promo_discount_query": [r"\bpromo code\b", r"\bdiscount\b", r"\bcoupon\b", r"\boffer\b"],
    "feature_request": [r"\bwould be (cool|nice|great)\b", r"\bit'?d be (cool|nice)\b", r"\bsuggestion\b", r"\bfeature\b"],
    "product_issue": [r"\bdamaged\b", r"\bbroken\b", r"\bdefective\b", r"\bnot working\b", r"\bfaulty\b"],
    "customer_service_complaint": [r"\bcustomer service\b", r"\brude\b", r"\bworst\b", r"\bterrible\b", r"\bdisappointed\b", r"\bhorrible\b"],
    "human_assistance_request": [r"\bhuman\b", r"\breal person\b", r"\bspeak to\b", r"\bcall me\b", r"\brepresentative\b"],
}

def get_matching_categories(text, patterns_dict):
    text_low = text.lower()
    matches = []
    for name, patterns in patterns_dict.items():
        if any(re.search(p, text_low) for p in patterns):
            matches.append(name)
    return matches

df_dev["candidate_labels"] = df_dev["text"].apply(lambda t: get_matching_categories(t, taxonomy_patterns))
df_dev["weak_label"] = df_dev["candidate_labels"].apply(lambda x: x[0] if x else "UNKNOWN")

print(df_dev["weak_label"].value_counts())

weak_label
delivery_status               24677
UNKNOWN                       14138
order_issue                    6953
device_tech_support            2957
refund_return                  2466
prime_membership               2401
customer_service_complaint     1667
payment_billing                1536
account_access                 1507
pricing_query                   517
product_issue                   442
content_streaming_issue         384
availability_question           383
promo_discount_query            196
human_assistance_request        136
feature_request                  73
repair_service_status            25
Name: count, dtype: int64


In [5]:
df_train = df_dev[df_dev["weak_label"] != "UNKNOWN"].copy()
print(f"Training examples (non-UNKNOWN): {len(df_train)}")
print(df_train["weak_label"].value_counts())

Training examples (non-UNKNOWN): 46320
weak_label
delivery_status               24677
order_issue                    6953
device_tech_support            2957
refund_return                  2466
prime_membership               2401
customer_service_complaint     1667
payment_billing                1536
account_access                 1507
pricing_query                   517
product_issue                   442
content_streaming_issue         384
availability_question           383
promo_discount_query            196
human_assistance_request        136
feature_request                  73
repair_service_status            25
Name: count, dtype: int64


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import joblib

X_train = df_train["text"]
y_train = df_train["weak_label"]

simple_baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

print("Training simple baseline...")
simple_baseline.fit(X_train, y_train)
print("Done!")

# Model save kar do (baad me evaluation harness me reuse karne ke liye)
import os
os.makedirs("../evaluation/baselines", exist_ok=True)
joblib.dump(simple_baseline, "../evaluation/baselines/simple_baseline_tfidf_logreg.pkl")
print("Saved model to evaluation/baselines/simple_baseline_tfidf_logreg.pkl")

Training simple baseline...
Done!
Saved model to evaluation/baselines/simple_baseline_tfidf_logreg.pkl


In [8]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# Golden set ko dataframe me convert karo
df_golden = pd.DataFrame(golden_set)

# NOTE: UNKNOWN ko bhi ek valid label maanenge (system ko UNKNOWN predict karna aana chahiye)
X_test = df_golden["current_message"]
y_true = df_golden["gold_intent"]

print(f"Evaluation set size: {len(df_golden)}")
print(f"Unique gold labels: {y_true.nunique()}")

Evaluation set size: 211
Unique gold labels: 18


In [9]:
y_pred_trivial = [trivial_baseline_predict(t) for t in X_test]

acc_trivial = accuracy_score(y_true, y_pred_trivial)
precision_trivial, recall_trivial, f1_trivial, _ = precision_recall_fscore_support(
    y_true, y_pred_trivial, average="macro", zero_division=0
)
_, _, weighted_f1_trivial, _ = precision_recall_fscore_support(
    y_true, y_pred_trivial, average="weighted", zero_division=0
)

print("=== TRIVIAL BASELINE ===")
print(f"Accuracy: {acc_trivial:.3f}")
print(f"Macro Precision: {precision_trivial:.3f}")
print(f"Macro Recall: {recall_trivial:.3f}")
print(f"Macro F1: {f1_trivial:.3f}")
print(f"Weighted F1: {weighted_f1_trivial:.3f}")

=== TRIVIAL BASELINE ===
Accuracy: 0.118
Macro Precision: 0.007
Macro Recall: 0.056
Macro F1: 0.012
Weighted F1: 0.025


In [10]:
# NOTE: model UNKNOWN label pe train nahi hua tha, isliye wo kabhi UNKNOWN predict nahi karega
# Ye ek IMPORTANT LIMITATION hai jo hum note karenge
y_pred_simple = simple_baseline.predict(X_test)

acc_simple = accuracy_score(y_true, y_pred_simple)
precision_simple, recall_simple, f1_simple, _ = precision_recall_fscore_support(
    y_true, y_pred_simple, average="macro", zero_division=0
)
_, _, weighted_f1_simple, _ = precision_recall_fscore_support(
    y_true, y_pred_simple, average="weighted", zero_division=0
)

print("=== SIMPLE BASELINE (TF-IDF + Logistic Regression) ===")
print(f"Accuracy: {acc_simple:.3f}")
print(f"Macro Precision: {precision_simple:.3f}")
print(f"Macro Recall: {recall_simple:.3f}")
print(f"Macro F1: {f1_simple:.3f}")
print(f"Weighted F1: {weighted_f1_simple:.3f}")

print("\n--- Detailed classification report ---")
print(classification_report(y_true, y_pred_simple, zero_division=0))

=== SIMPLE BASELINE (TF-IDF + Logistic Regression) ===
Accuracy: 0.502
Macro Precision: 0.475
Macro Recall: 0.477
Macro F1: 0.461
Weighted F1: 0.509

--- Detailed classification report ---
                            precision    recall  f1-score   support

                   UNKNOWN       0.00      0.00      0.00        18
            account_access       0.75      0.92      0.83        13
     availability_question       0.56      0.42      0.48        12
   content_streaming_issue       0.62      0.38      0.48        13
customer_service_complaint       0.53      0.64      0.58        14
           delivery_status       0.60      0.60      0.60        25
       device_tech_support       0.23      0.60      0.33         5
           feature_request       0.50      0.40      0.44        10
  human_assistance_request       0.40      0.57      0.47         7
               order_issue       0.16      0.25      0.19        12
        packaging_feedback       0.00      0.00      0.00     

In [11]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"],
    "Trivial Baseline": [acc_trivial, precision_trivial, recall_trivial, f1_trivial, weighted_f1_trivial],
    "Simple Baseline": [acc_simple, precision_simple, recall_simple, f1_simple, weighted_f1_simple],
})
print(comparison.to_string(index=False))

         Metric  Trivial Baseline  Simple Baseline
       Accuracy          0.118483         0.502370
Macro Precision          0.006582         0.475436
   Macro Recall          0.055556         0.476965
       Macro F1          0.011770         0.461290
    Weighted F1          0.025102         0.508981


In [12]:
baseline_results = {
    "trivial_baseline": {
        "accuracy": acc_trivial, "macro_precision": precision_trivial,
        "macro_recall": recall_trivial, "macro_f1": f1_trivial, "weighted_f1": weighted_f1_trivial,
    },
    "simple_baseline": {
        "accuracy": acc_simple, "macro_precision": precision_simple,
        "macro_recall": recall_simple, "macro_f1": f1_simple, "weighted_f1": weighted_f1_simple,
        "note": "Never predicts UNKNOWN — trained only on non-UNKNOWN weak-labeled data",
    },
}

with open("../evaluation/reports/baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline results saved to evaluation/reports/baseline_results.json")

Baseline results saved to evaluation/reports/baseline_results.json
